# Lab 3.1: Agentes com o Agent Service do Azure AI Foundry (20 min)

## Objetivos
- Criar um **agente persistente** no Azure AI Foundry (Assistants API)
- Entender o ciclo: **Agent → Thread → Message → Run**
- Usar **function calling** com agentes
- Usar o **Code Interpreter**
- Gerir o ciclo de vida do agente (criar, conversar, eliminar)

## Conceitos

### Agente vs Chamada ao Modelo
No Lab 3 usámos a **Responses API** — chamadas diretas ao modelo. Aqui vamos criar **agentes persistentes**:

| Responses API (Lab 3) | Agent Service (Lab 3.1) |
|---|---|
| Chamada stateless | Agente persistente |
| Sem memória entre chamadas | Threads mantêm contexto |
| Tu geres o loop de tools | O Run processa tools automaticamente |
| Ideal para uso simples | Ideal para assistentes complexos |

### Ciclo de Vida
```
1. Criar Agente (com instruções e tools)
2. Criar Thread (conversa)
3. Adicionar Mensagem ao Thread
4. Criar Run (execução)
5. Processar Resposta (pode incluir tool calls)
6. Repetir 3-5 para continuar a conversa
7. Eliminar Agente quando já não for necessário
```

In [1]:
# Setup
from dotenv import load_dotenv
import os
import json

load_dotenv("../.env")

from openai import AzureOpenAI

endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
key = os.getenv("AZURE_AI_FOUNDRY_KEY")
model = os.getenv("MODEL_DEPLOYMENT", "gpt-4o")

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=key,
    api_version="2025-03-01-preview",
)

print(f"\u2705 Cliente conectado: {endpoint}")

✅ Cliente conectado: https://foundry-mod2.cognitiveservices.azure.com/


## 🖥️ Exercício 3.1.1: Agente Simples

Vamos criar um agente persistente com instruções e conversar com ele através de **threads**.

In [2]:
# Exercício 3.1.1: Criar um agente persistente
agent = client.beta.assistants.create(
    model=model,
    name="assistente-azure",
    instructions="És um especialista em Azure. Responde de forma concisa em português de Portugal. Usa exemplos práticos.",
)
print(f"\u2705 Agente criado: {agent.id}")
print(f"   Nome: {agent.name}")
print(f"   Modelo: {agent.model}")

/tmp/ipykernel_13617/63555087.py:2: DeprecationWarning: deprecated
  agent = client.beta.assistants.create(


✅ Agente criado: asst_ntuUhsgbfgkgB7Xg48l9caPf
   Nome: assistente-azure
   Modelo: gpt-4o


In [3]:
# Criar thread (conversa) e enviar primeira mensagem
thread = client.beta.threads.create()
print(f"\u2705 Thread criada: {thread.id}")

client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="Explica a diferença entre Azure Functions e Container Apps em 3 linhas.",
)

# Executar o agente no thread
run = client.beta.threads.runs.create_and_poll(
    thread_id=thread.id,
    assistant_id=agent.id,
)
print(f"   Run status: {run.status}")

# Obter resposta
messages = client.beta.threads.messages.list(thread_id=thread.id)
for msg in reversed(list(messages)):
    role = "\U0001f464" if msg.role == "user" else "\U0001f916"
    for block in msg.content:
        if block.type == "text":
            print(f"{role} {block.text.value}")

/tmp/ipykernel_13617/3876056952.py:2: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  thread = client.beta.threads.create()


✅ Thread criada: thread_S4q0YrwtBKpm58sN74v6JZoP


/tmp/ipykernel_13617/3876056952.py:5: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  client.beta.threads.messages.create(
/tmp/ipykernel_13617/3876056952.py:12: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.create_and_poll(


   Run status: completed


/tmp/ipykernel_13617/3876056952.py:19: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  messages = client.beta.threads.messages.list(thread_id=thread.id)


👤 Explica a diferença entre Azure Functions e Container Apps em 3 linhas.
🤖 Azure Functions é um serviço serverless para executar pequenas unidades de código em resposta a eventos, ideal para tarefas automatizadas e escaláveis. Container Apps permite executar aplicações em contêineres com maior controle, ótimo para microserviços complexos. Use Functions para simplicidade e Container Apps para maior flexibilidade e personalização (ex.: APIs ou workers).


In [4]:
# Continuar a conversa no MESMO thread (o agente mantém contexto!)
client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="E qual dos dois recomendas para um webhook simples?",
)

run2 = client.beta.threads.runs.create_and_poll(
    thread_id=thread.id,
    assistant_id=agent.id,
)
print(f"Run status: {run2.status}\n")

# Mostrar toda a conversa
messages = client.beta.threads.messages.list(thread_id=thread.id)
for msg in reversed(list(messages)):
    role = "\U0001f464" if msg.role == "user" else "\U0001f916"
    for block in msg.content:
        if block.type == "text":
            print(f"{role} {block.text.value}\n")

/tmp/ipykernel_13617/3683772741.py:2: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  client.beta.threads.messages.create(
/tmp/ipykernel_13617/3683772741.py:8: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run2 = client.beta.threads.runs.create_and_poll(


Run status: completed



/tmp/ipykernel_13617/3683772741.py:15: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  messages = client.beta.threads.messages.list(thread_id=thread.id)


👤 Explica a diferença entre Azure Functions e Container Apps em 3 linhas.

🤖 Azure Functions é um serviço serverless para executar pequenas unidades de código em resposta a eventos, ideal para tarefas automatizadas e escaláveis. Container Apps permite executar aplicações em contêineres com maior controle, ótimo para microserviços complexos. Use Functions para simplicidade e Container Apps para maior flexibilidade e personalização (ex.: APIs ou workers).

👤 E qual dos dois recomendas para um webhook simples?

🤖 Recomendo Azure Functions para um webhook simples, pois é serverless, fácil de configurar e paga apenas pelo tempo de execução. Um exemplo seria criar uma Function HTTP Trigger para processar as solicitações do webhook de forma eficiente.



## 🖥️ Exercício 3.1.2: Agente com Function Calling

O verdadeiro poder dos agentes é usar **ferramentas**. Vamos criar um agente que pode consultar informações sobre preços e regiões Azure.

In [5]:
# Exercício 3.1.2: Definir funções e criar agente com tools

# Funções que o agente pode chamar
def obter_preco_servico(servico: str) -> str:
    """Retorna o preço estimado de um serviço Azure."""
    precos = {
        "app service basic": "\u20ac50/mês",
        "container apps": "\u20ac0.000012/vCPU-s",
        "functions consumption": "\u20ac0.000016/GB-s",
        "aks": "Grátis (control plane) + custo dos nodes",
        "cosmos db": "A partir de \u20ac25/mês (serverless)",
    }
    return json.dumps({"servico": servico, "preco": precos.get(servico.lower(), "Preço não disponível. Consulta azure.com/pricing")})

def obter_regiao_recomendada(caso_uso: str) -> str:
    """Recomenda a melhor região Azure para um caso de uso."""
    regioes = {
        "europa": "West Europe (Holanda) ou North Europe (Irlanda)",
        "portugal": "West Europe (mais próximo de Portugal)",
        "global": "Usa Traffic Manager com múltiplas regiões",
        "ai": "East US ou Sweden Central (melhor disponibilidade de modelos)",
    }
    return json.dumps({"caso_uso": caso_uso, "recomendacao": regioes.get(caso_uso.lower(), regioes["europa"])})

# Mapa de funções disponíveis
available_functions = {
    "obter_preco_servico": obter_preco_servico,
    "obter_regiao_recomendada": obter_regiao_recomendada,
}

# Criar agente com tools
tools = [
    {"type": "function", "function": {
        "name": "obter_preco_servico",
        "description": "Obtém o preço estimado de um serviço Azure",
        "parameters": {
            "type": "object",
            "properties": {"servico": {"type": "string", "description": "Nome do serviço Azure (ex: app service basic, container apps)"}},
            "required": ["servico"]
        }
    }},
    {"type": "function", "function": {
        "name": "obter_regiao_recomendada",
        "description": "Recomenda a melhor região Azure para um caso de uso",
        "parameters": {
            "type": "object",
            "properties": {"caso_uso": {"type": "string", "description": "Caso de uso (europa, portugal, global, ai)"}},
            "required": ["caso_uso"]
        }
    }},
]

agent_tools = client.beta.assistants.create(
    model=model,
    name="consultor-azure",
    instructions="És um consultor Azure. Usa as ferramentas disponíveis para dar informações precisas sobre preços e regiões. Responde em português.",
    tools=tools,
)
print(f"\u2705 Agente com tools criado: {agent_tools.id}")

/tmp/ipykernel_13617/3022204622.py:53: DeprecationWarning: deprecated
  agent_tools = client.beta.assistants.create(


✅ Agente com tools criado: asst_mxhEiLeQXKyyx2MIxUkBFvGu


In [6]:
# Testar o agente com function calling
thread2 = client.beta.threads.create()
client.beta.threads.messages.create(
    thread_id=thread2.id,
    role="user",
    content="Quero criar uma app com Container Apps. Quanto custa e qual a melhor região para Portugal?",
)

# Criar run
run3 = client.beta.threads.runs.create_and_poll(
    thread_id=thread2.id,
    assistant_id=agent_tools.id,
)
print(f"Run status: {run3.status}")

# Processar tool calls se necessário
while run3.status == "requires_action":
    tool_outputs = []
    for call in run3.required_action.submit_tool_outputs.tool_calls:
        func = available_functions[call.function.name]
        args = json.loads(call.function.arguments)
        result = func(**args)
        print(f"   \U0001f527 {call.function.name}({args}) \u2192 {result}")
        tool_outputs.append({"tool_call_id": call.id, "output": result})
    
    run3 = client.beta.threads.runs.submit_tool_outputs_and_poll(
        thread_id=thread2.id,
        run_id=run3.id,
        tool_outputs=tool_outputs,
    )
    print(f"   Run status: {run3.status}")

# Mostrar resultado
messages2 = client.beta.threads.messages.list(thread_id=thread2.id)
for msg in reversed(list(messages2)):
    role = "\U0001f464" if msg.role == "user" else "\U0001f916"
    for block in msg.content:
        if block.type == "text":
            print(f"{role} {block.text.value}\n")

/tmp/ipykernel_13617/2387923143.py:2: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  thread2 = client.beta.threads.create()
/tmp/ipykernel_13617/2387923143.py:3: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  client.beta.threads.messages.create(
/tmp/ipykernel_13617/2387923143.py:10: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run3 = client.beta.threads.runs.create_and_poll(


Run status: requires_action
   🔧 obter_preco_servico({'servico': 'container apps'}) → {"servico": "container apps", "preco": "\u20ac0.000012/vCPU-s"}
   🔧 obter_regiao_recomendada({'caso_uso': 'portugal'}) → {"caso_uso": "portugal", "recomendacao": "West Europe (mais pr\u00f3ximo de Portugal)"}


/tmp/ipykernel_13617/2387923143.py:26: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run3 = client.beta.threads.runs.submit_tool_outputs_and_poll(


   Run status: completed


/tmp/ipykernel_13617/2387923143.py:34: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  messages2 = client.beta.threads.messages.list(thread_id=thread2.id)


👤 Quero criar uma app com Container Apps. Quanto custa e qual a melhor região para Portugal?

🤖 O custo estimado para utilizar o serviço de Container Apps no Azure é de **€0.000012 por segundo de uso da vCPU**. 

A região mais recomendada para implantações em Portugal é a **West Europe**, que é a mais próxima e garantirá melhor desempenho para os seus serviços.



## 🖥️ Exercício 3.1.3: Code Interpreter

O **Code Interpreter** permite ao agente escrever e executar código Python autonomamente.

In [7]:
# Exercício 3.1.3: Agente com Code Interpreter
agent_code = client.beta.assistants.create(
    model=model,
    name="analista-dados",
    instructions="És um analista de dados. Usa o code interpreter para fazer cálculos e criar visualizações. Responde em português.",
    tools=[{"type": "code_interpreter"}],
)
print(f"\u2705 Agente com code interpreter: {agent_code.id}")

thread3 = client.beta.threads.create()
client.beta.threads.messages.create(
    thread_id=thread3.id,
    role="user",
    content="Calcula a sequência de Fibonacci até ao 10º número e mostra o resultado numa tabela formatada.",
)

run4 = client.beta.threads.runs.create_and_poll(
    thread_id=thread3.id,
    assistant_id=agent_code.id,
)
print(f"   Run status: {run4.status}\n")

messages3 = client.beta.threads.messages.list(thread_id=thread3.id)
for msg in reversed(list(messages3)):
    if msg.role == "assistant":
        for block in msg.content:
            if block.type == "text":
                print(f"\U0001f916 {block.text.value}")

/tmp/ipykernel_13617/353462080.py:2: DeprecationWarning: deprecated
  agent_code = client.beta.assistants.create(


✅ Agente com code interpreter: asst_iML4wOAaACJtnVGE9K46jrD3


/tmp/ipykernel_13617/353462080.py:10: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  thread3 = client.beta.threads.create()
/tmp/ipykernel_13617/353462080.py:11: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  client.beta.threads.messages.create(
/tmp/ipykernel_13617/353462080.py:17: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run4 = client.beta.threads.runs.create_and_poll(


   Run status: completed



/tmp/ipykernel_13617/353462080.py:23: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  messages3 = client.beta.threads.messages.list(thread_id=thread3.id)


🤖 Aqui está a tabela com a sequência de Fibonacci até ao 10º número:

| Posição (n) | Fibonacci (F(n)) |
|-------------|------------------|
| 1           | 0                |
| 2           | 1                |
| 3           | 1                |
| 4           | 2                |
| 5           | 3                |
| 6           | 5                |
| 7           | 8                |
| 8           | 13               |
| 9           | 21               |
| 10          | 34               |


## 🧹 Limpeza

Os agentes são **persistentes** — é importante eliminá-los quando já não são necessários.

In [8]:
# Limpeza - eliminar todos os agentes criados
for a in [agent, agent_tools, agent_code]:
    client.beta.assistants.delete(a.id)
    print(f"\U0001f5d1\ufe0f Agente {a.name} eliminado ({a.id})")

print("\n\u2705 Limpeza concluída! Todos os agentes foram eliminados.")

/tmp/ipykernel_13617/3459531829.py:3: DeprecationWarning: deprecated
  client.beta.assistants.delete(a.id)


🗑️ Agente assistente-azure eliminado (asst_ntuUhsgbfgkgB7Xg48l9caPf)
🗑️ Agente consultor-azure eliminado (asst_mxhEiLeQXKyyx2MIxUkBFvGu)
🗑️ Agente analista-dados eliminado (asst_iML4wOAaACJtnVGE9K46jrD3)

✅ Limpeza concluída! Todos os agentes foram eliminados.


## ✅ Resumo

Neste lab aprendeste:
- Criar **agentes persistentes** com o Agent Service do Foundry
- O ciclo **Agent → Thread → Message → Run**
- Manter **contexto** entre mensagens usando threads
- Usar **function calling** com o loop de tool calls
- Usar o **Code Interpreter** para computação autónoma
- Gerir o ciclo de vida (criar e eliminar agentes)

### Lab 3 vs Lab 3.1
| Lab 3 (Responses API) | Lab 3.1 (Agent Service) |
|---|---|
| Chamadas diretas ao modelo | Agentes persistentes |
| Tu geres o contexto | Threads mantêm o histórico |
| Tu processas tool calls | O Run processa automaticamente |
| Mais simples e leve | Mais poderoso para assistentes complexos |

**Próximo:** [Lab 4 - Knowledge e RAG](lab04-knowledge-rag.ipynb)